# Parte 1: Configuración y preparación del VectorStore 

In [ ]:
# Load config and build embeddings/vectorstore factories
import json
from pathlib import Path

from modules.utils.models import LangChainModelFactory
from modules.utils.vectorstores import LangChainVectorStoreFactory

# Resolve config path from either workspace root or notebook folder
candidate_paths = [
    Path("./config/config_chroma.json"),
    Path("config/config_chroma.json"),
]
config_path = next((p for p in candidate_paths if p.exists()), None)
if config_path is None:
    raise FileNotFoundError("config_chroma.json not found in expected paths.")

with config_path.open("r", encoding="utf-8") as f:
    config_json = json.load(f)

# Build embeddings and llm model from config
model_factory = LangChainModelFactory(config_json["models"])
embeddings_model = model_factory.build_embeddings()

llm_config = config_json["models"]
llm_factory = LangChainModelFactory(llm_config)
llm = llm_factory.build_llm()
print("Embeddings model:", embeddings_model.__class__.__name__)
print("LLM model:", llm.__class__.__name__)

# Build vectorstore from config
vectorstore_factory = LangChainVectorStoreFactory(
    config=config_json["vectorstore"],
    embeddings=embeddings_model,
)
vector_store = vectorstore_factory.build()

print("Config loaded from:", config_path)
print("Embeddings model:", embeddings_model.__class__.__name__)
print("Vector store:", vector_store.__class__.__name__)

## Parte 2 Ingesta de Documentos

In [ ]:
# Funciones auxiliares para procesamiento de PDFs y texto. Variables de configuración del proceso
from pathlib import Path
import re
import html
import hashlib
import fitz

from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)

PDF_ROOT = Path("./data/pdf")
HTML_OUTPUT_DIR = PDF_ROOT / "html_pages"
HTML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If True, rebuilds the collection from scratch to avoid duplicated vectors between runs.
CLEAN_START = True
BATCH_SIZE = 200

if not PDF_ROOT.exists():
    raise FileNotFoundError(f"PDF root folder not found: {PDF_ROOT}")

if CLEAN_START:
    try:
        vector_store.delete_collection()
        vector_store = vectorstore_factory.build()
        print("Collection reset completed.")
    except Exception as e:
        print(f"Warning: could not reset collection ({e}). Continuing with append mode.")


def strip_html_tags(raw_html: str) -> str:
    text = re.sub(r"<[^>]+>", " ", raw_html)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def infer_section_name_from_html(page_html: str, page_text: str) -> str:
    header_tag_pattern = re.compile(r"<(h[1-6])[^>]*>(.*?)</\1>", re.IGNORECASE | re.DOTALL)
    headers = [strip_html_tags(m.group(2)) for m in header_tag_pattern.finditer(page_html)]
    headers = [h for h in headers if len(h) >= 3]
    if headers:
        return headers[0][:180]

    font_pattern = re.compile(
        r"<(?:span|p|div)[^>]*style=\"[^\"]*font-size\s*:\s*([0-9]+(?:\.[0-9]+)?)pt[^\"]*\"[^>]*>(.*?)</(?:span|p|div)>",
        re.IGNORECASE | re.DOTALL,
    )
    candidates = []
    for m in font_pattern.finditer(page_html):
        size = float(m.group(1))
        txt = strip_html_tags(m.group(2))
        if txt and len(txt) >= 3:
            candidates.append((size, txt))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        return candidates[0][1][:180]

    fallback_lines = [ln.strip() for ln in re.split(r"(?<=[\.!?])\s+", page_text) if ln.strip()]
    return fallback_lines[0][:180] if fallback_lines else "Sin sección"


def build_text_splitter(config_json: dict):
    chunking_cfg = config_json.get("vectorstore", {}).get("chunking", {})
    chunk_type = str(chunking_cfg.get("type", "recursive")).strip().lower()
    chunk_size = int(chunking_cfg.get("chunk_size", 1200))
    chunk_overlap = int(chunking_cfg.get("chunk_overlap", 150))

    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    if chunk_type == "recursive":
        return RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=chunking_cfg.get("separators"),
        )

    if chunk_type == "character":
        return CharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separator=chunking_cfg.get("separator", "\n\n"),
        )

    if chunk_type == "token":
        # Uses tiktoken under the hood if available.
        return TokenTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    raise ValueError(
        f"Unsupported chunking.type: {chunk_type}. Use one of: recursive, character, token"
    )


## Proceso de ingesta
Ingestar los ficheros que aparezcan en el path seleccionado de PDFs

In [ ]:
pdf_files = sorted(PDF_ROOT.rglob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found recursively under: {PDF_ROOT}")

splitter = build_text_splitter(config_json)
print(
    "Chunking config:",
    config_json.get("vectorstore", {}).get("chunking", {"type": "recursive", "chunk_size": 1200, "chunk_overlap": 150}),
)

doc_batch = []
id_batch = []

stats = {
    "pdf_files": 0,
    "pages": 0,
    "chunks": 0,
}

for pdf_path in pdf_files:
    # categoria = top-level folder under root path or "sin_categoria" if PDF is directly under root
    rel_path = pdf_path.relative_to(PDF_ROOT)
    categoria = rel_path.parts[0] if len(rel_path.parts) > 1 else "sin_categoria"

    stats["pdf_files"] += 1

    with fitz.open(str(pdf_path)) as pdf:
        total_pages = pdf.page_count
        stats["pages"] += total_pages

        for idx in range(total_pages):
            page = pdf.load_page(idx)
            page_html = page.get_text("html")

            html_file = HTML_OUTPUT_DIR / f"{pdf_path.stem}_page_{idx + 1}.html"
            html_file.write_text(page_html, encoding="utf-8")

            page_text = strip_html_tags(page_html)
            section_name = infer_section_name_from_html(page_html, page_text)
            page_number = idx + 1
            page_name = f"{pdf_path.name} - page {page_number}"

            chunks = splitter.split_text(page_text)
            for chunk_id, chunk in enumerate(chunks, start=1):
                metadata = {
                    "categoria": categoria,
                    "source_file": pdf_path.name,
                    "source_path": str(pdf_path),
                    "page_number": page_number,
                    "page_name": page_name,
                    "section_name": section_name,
                    "chunk_id": chunk_id,
                    "page_html_path": str(html_file),
                }

                doc = Document(page_content=chunk, metadata=metadata)
                doc_batch.append(doc)

                # Deterministic id to support reproducible re-ingestion
                id_raw = f"{pdf_path}|{page_number}|{chunk_id}|{categoria}"
                id_batch.append(hashlib.sha1(id_raw.encode("utf-8")).hexdigest())
                stats["chunks"] += 1

                if len(doc_batch) >= BATCH_SIZE:
                    vector_store.add_documents(documents=doc_batch, ids=id_batch)
                    doc_batch, id_batch = [], []

# Flush remaining batch
if doc_batch:
    vector_store.add_documents(documents=doc_batch, ids=id_batch)

print("Ingestion completed")
print("PDF files processed:", stats["pdf_files"])
print("Pages processed:", stats["pages"])
print("Chunks indexed:", stats["chunks"])
print("Collection:", config_json["vectorstore"]["chroma"]["collection_name"])
print("Persist directory:", config_json["vectorstore"]["chroma"].get("persist_directory"))


# Parte 3: Preparación de RAG (prompts, orquestador LLM y adaptar respuesta)

In [ ]:
# Build prompts dict from ./data/prompts
import json
from pathlib import Path

prompts_dir = Path("./data/prompts")
if not prompts_dir.exists():
    raise FileNotFoundError(f"Prompts folder not found: {prompts_dir}")

# Mapping required by orchestrator nodes -> prompt file stem
node_prompt_file_map = {
    "AgentNode": "prompt_agent",
    "GradeDocumentsNode": "prompt_grade_documents",
    "GenerateNode": "prompt_generate",
    "ChangeQuestionNode": "prompt_transform_query",
    "RewriteNode": "prompt_rewrite",
    "SuggestedQuestionsNode": "prompt_suggested_questions",
}

# Optional validation against prompt_guide.json (if present)
prompt_guide_path = prompts_dir / "prompt_guide.json"
if prompt_guide_path.exists():
    with prompt_guide_path.open("r", encoding="utf-8") as f:
        prompt_guide = json.load(f)
    guide_values = set(prompt_guide.values())
    missing_from_guide = set(node_prompt_file_map.values()) - guide_values
    if missing_from_guide:
        print("Warning: Some prompt files are not referenced in prompt_guide.json:", sorted(missing_from_guide))

prompts = {}
missing_files = []

for node_name, prompt_stem in node_prompt_file_map.items():
    prompt_path = prompts_dir / f"{prompt_stem}.txt"
    if not prompt_path.exists():
        missing_files.append(str(prompt_path))
        continue

    prompt_text = prompt_path.read_text(encoding="utf-8").strip()
    prompts[node_name] = {
        "text": prompt_text,
        "source": str(prompt_path),
    }

if missing_files:
    raise FileNotFoundError(
        "Missing prompt files required by orchestrator:\n- " + "\n- ".join(missing_files)
    )

print("Prompts loaded:", sorted(prompts.keys()))
print("Total prompts:", len(prompts))

In [ ]:
def build_public_response(output: dict) -> dict:
    # 1) Respuesta final
    respuesta = ""
    for msg in reversed(output.get("messages", [])):
        content = getattr(msg, "content", "")
        if isinstance(content, str) and content.strip():
            respuesta = content.strip()
            break

    # 2) Referencias desde context
    referencias = []
    seen = set()
    for i, doc in enumerate(output.get("context", []), 1):
        md = doc.metadata or {}
        key = (
            md.get("source_path"),
            md.get("page_number"),
            md.get("chunk_id"),
        )
        if key in seen:
            continue
        seen.add(key)

        referencias.append({
            "id": f"ref_{len(referencias)+1}",
            "documento": md.get("source_file"),
            "categoria": md.get("categoria"),
            "seccion": md.get("section_name"),
            "pagina": md.get("page_number"),
            "page_name": md.get("page_name"),
            "source_path": md.get("source_path"),
        })

    return {
        "respuesta": respuesta,
        "referencias": referencias,
        "preguntas_sugeridas": output.get("suggested_questions", []),
        "filtros_aplicados": {
            "categoria": output.get("categoria"),
            "idiomas": output.get("languages", []),
        },
        "meta": {
            "total_referencias": len(referencias),
            "agent_n_calls": output.get("agent_n_calls", 0),
        },
    }

In [ ]:
from langchain_core.tools import tool


# Tool for the agent to launch the retriever
@tool(response_format='content_and_artifact')
def retriever_tool(queries: list[str], categoria: str | None = None):
    '''
    Busca información para resolver dudas de los documentos.
    Soporta filtro opcional por categoria (nombre de carpeta: legal, informes, manuales, etc.).
    '''
    vectorstore_provider = config_json.get("vectorstore", {}).get("provider")

    if vectorstore_provider != "chroma":
        raise ValueError(f"Unsupported vector store provider: {vectorstore_provider}")

    k = config_json.get("retriever", {}).get("search_kwargs", {}).get("k", 4)

    # Base search kwargs
    search_kwargs = {"k": k}

    # Optional category filter over ingested metadata
    if categoria:
        search_kwargs["filter"] = {"categoria": categoria}

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs=search_kwargs,
    )

    retrieved_docs = []
    print(f"Queries for retriever: {queries}")
    print(f"Retriever search kwargs: {search_kwargs}")

    for query in queries:
        retrieved_docs_tmp = retriever.invoke(query)
        retrieved_docs.extend(retrieved_docs_tmp)

    if len(retrieved_docs) == 0:
        print("No documents retrieved.")
    else:
        print(f"Retrieved {len(retrieved_docs)} documents.")
        # Preview metadata so it is easier to debug retrieval behavior
        for i, doc in enumerate(retrieved_docs[:5], 1):
            md = doc.metadata or {}
            print(
                f"Doc {i}: categoria={md.get('categoria')} | "
                f"section={md.get('section_name')} | page={md.get('page_name')}"
            )

    serialized = '\n\n'.join(
        (f"Source: {doc.metadata}\n" f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )

    return serialized, retrieved_docs if retrieved_docs is not None else []

In [ ]:
from modules.application.orchestrator_a import Orchestrator
from pathlib import Path
import json

# Build orchestrator once
orchestrator = Orchestrator(
    retriever_tool=retriever_tool,
    prompts=prompts,
    model=llm,
    config_params=config_json,
).compile()

# I/O paths
input_path = Path("./tests/input.json")
output_path = Path("./tests/output.json")

if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path}")

with input_path.open("r", encoding="utf-8") as f:
    test_inputs = json.load(f)

if not isinstance(test_inputs, list):
    raise ValueError("Input JSON must be a list of test cases.")

results = []

for idx, item in enumerate(test_inputs, start=1):
    question = (item.get("question") or "").strip()
    # TODO modificar la extracción de este campo desde el JSON por un proceso de detección automática (p.ej.: modelo transformer) de categoría basado en el contenido de la pregunta o metadatos asociados pasando un listado de opciones (el de las carpetas del PDF)
    categoria = item.get("categoria")
    # TODO: alternativamente, añadir un nodo al orquestador para detectar categoría automáticamente a partir de la pregunta o metadatos asociados y eliminar esta dependencia en el input JSON
    # Keep expected fields for later DeepEval process
    expected_answer = item.get("expected_answer", "")
    expected_references = item.get("expected_references", [])

    if not question:
        # Preserve row with empty question to make data issues explicit
        results.append(
            {
                "test_id": idx,
                "question": question,
                "categoria": categoria,
                "expected_answer": expected_answer,
                "expected_references": expected_references,
                "actual_output": "",
                "actual_references": [],
                "status": "skipped_empty_question",
                "error": "Question is empty",
            }
        )
        continue

    print(f"[{idx}/{len(test_inputs)}] Invoking orchestrator for question: {question}")

    try:
        output = orchestrator.invoke(
            {
                "messages": [question],
                "languages": ["spanish"],
                "categoria": categoria,
            }
        )

        public_output = build_public_response(output)

        # Normalize references for evaluation format
        actual_references = []
        for ref in public_output.get("referencias", []):
            actual_references.append(
                {
                    "documento": ref.get("documento"),
                    "pagina": ref.get("pagina"),
                    "seccion": ref.get("seccion"),
                    "categoria": ref.get("categoria"),
                    "source_path": ref.get("source_path"),
                }
            )

        results.append(
            {
                "test_id": idx,
                "question": question,
                "categoria": categoria,
                "expected_answer": expected_answer,
                "expected_references": expected_references,
                "actual_output": public_output.get("respuesta", ""),
                "actual_references": actual_references,
                "suggested_questions": public_output.get("preguntas_sugeridas", []),
                "status": "ok",
                "error": None,
            }
        )

    except Exception as e:
        results.append(
            {
                "test_id": idx,
                "question": question,
                "categoria": categoria,
                "expected_answer": expected_answer,
                "expected_references": expected_references,
                "actual_output": "",
                "actual_references": [],
                "status": "error",
                "error": str(e),
            }
        )

# Write output file for evaluation pipeline
with output_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nProcessed test cases: {len(results)}")
print(f"Output written to: {output_path}")


# Evaluación

## Preparación de métricas deepeval

In [ ]:
import openai
import time
from deepeval.models.llms import AzureOpenAIModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import GEval, AnswerRelevancyMetric, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import SingleTurnParams


if config_json["models"]["provider"] == "azure_openai":
    
    model = AzureOpenAIModel(
        model=config_json["models"]["azure_openai"]["llm"]["deployment_name"],
        deployment_name=config_json["models"]["azure_openai"]["llm"]["deployment_name"],
        api_key=config_json["models"]["azure_openai"]["api_key"],
        base_url=config_json["models"]["azure_openai"]["endpoint"],
        api_version=config_json["models"]["azure_openai"]["api_version"],
        generation_kwargs={
            "max_completion_tokens": 4000  # Suficiente para gpt-4o
        }
    )

else:
    raise ValueError("Current LLM provider in config is not supported for evaluation. Please switch to 'azure_openai' and set the required env vars.")


correctness_metric = GEval(
    model=model,
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the expected output.",
    # NOTE: you can only provide either criteria or evaluation_steps, and not both
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradicts any facts in 'expected output'",
        "You should also heavily penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are OK"
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
)
metrics = [
    AnswerRelevancyMetric(model=model),
    FaithfulnessMetric(model=model),
    ContextualRelevancyMetric(model=model),
    correctness_metric,

]



## Proceso de evaluación

In [ ]:
from modules.utils.adapt_fields import build_testcase_from_output_row, compute_page_similarity, PAGE_TOLERANCE
from statistics import mean

output_path = Path("./tests/output.json")
eval_report_path = Path("./tests/eval_report.json")

if not output_path.exists():
    raise FileNotFoundError(f"Output file not found: {output_path}")

if "metrics" not in globals():
    raise RuntimeError("'metrics' not found. Run the metrics setup cell first.")

with output_path.open("r", encoding="utf-8") as f:
    rows = json.load(f)

if not isinstance(rows, list):
    raise ValueError("output.json must be a list of test rows.")

case_reports = []

for row in rows:
    test_id = row.get("test_id")
    status = row.get("status")

    if status != "ok":
        case_reports.append(
            {
                "test_id": test_id,
                "question": row.get("question"),
                "status": status,
                "error": row.get("error"),
                "metrics": [],
                "page_similarity": None,
            }
        )
        continue

    test_case = build_testcase_from_output_row(row)

    metric_results = []
    for metric in metrics:
        metric.measure(test_case)
        metric_results.append(
            {
                "name": getattr(metric, "name", metric.__class__.__name__),
                "score": metric.score,
                "passed": metric.success,
                "reason": metric.reason,
            }
        )

    page_similarity = compute_page_similarity(
        expected_refs=row.get("expected_references", []),
        actual_refs=row.get("actual_references", []),
        tolerance=PAGE_TOLERANCE,
    )

    case_reports.append(
        {
            "test_id": test_id,
            "question": row.get("question"),
            "categoria": row.get("categoria"),
            "status": "evaluated",
            "metrics": metric_results,
            "page_similarity": page_similarity,
        }
    )

# Aggregate summary over evaluated cases
evaluated = [c for c in case_reports if c.get("status") == "evaluated"]
summary = {
    "total_cases": len(rows),
    "evaluated_cases": len(evaluated),
    "skipped_or_error_cases": len(rows) - len(evaluated),
    "metrics": {},
    "page_similarity": {
        "avg_score": None,
    },
}

if evaluated:
    metric_names = [m.get("name") for m in evaluated[0]["metrics"]]
    for mname in metric_names:
        mvals = [
            m for c in evaluated for m in c["metrics"]
            if m.get("name") == mname
        ]
        summary["metrics"][mname] = {
            "avg_score": mean([mv["score"] for mv in mvals if mv["score"] is not None]) if mvals else None,
            "pass_rate": (
                sum(1 for mv in mvals if mv.get("passed")) / len(mvals)
                if mvals else None
            ),
        }

    page_scores = [
        c.get("page_similarity", {}).get("avg_page_similarity")
        for c in evaluated
        if c.get("page_similarity", {}).get("avg_page_similarity") is not None
    ]
    summary["page_similarity"]["avg_score"] = mean(page_scores) if page_scores else None

report = {
    "summary": summary,
    "cases": case_reports,
    "config": {
        "page_tolerance": PAGE_TOLERANCE,
    },
}

with eval_report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Evaluated cases: {summary['evaluated_cases']}/{summary['total_cases']}")
print(f"Evaluation report written to: {eval_report_path}")
print(f"Average page similarity: {summary['page_similarity']['avg_score']}")